# 02. Geospatial Entity Resolution, Cross-Source OSM Verification & B2B Contact Enrichment

Consumes the enriched stage-01 outputs (`denue_state_11_radius_5km.parquet`, `b2b_gold_list_5d_morning.csv`, `osm_amenities.parquet`), verifies physical storefronts and facilities against OpenStreetMap geometries, clusters multi-plant corporate campuses using NLP (RapidFuzz + branch stripping) while preventing false merges, unlocks the **32 high-priority targets missing email** and **Tier-1 partial accounts** via campus inheritance, web scraping/domain synthesis, and phone/LinkedIn ABM routing, and exports the final Stage-02 Production Contract.

### Stage-02 Production Architecture & Output Contract
1. **Canonical Account Master ([stage2_canonical_action_master.csv](tables/stage2_canonical_action_master.csv)):** Strictly deduplicated table (1 row per corporate parent) with consolidated headcount, payroll mass, and best verified outreach channel for sales and CRM outreach.
2. **Site-Level Routing Table ([stage2_site_action_list.csv](tables/stage2_site_action_list.csv)):** Physical establishment-level table (1 row per branch/plant) with geocoordinates, OSM verification distance, and route order for field visits.
3. **Contact Confidence & Provenance Hierarchy:** Explicit tracking of `email_source`, `email_confidence` (`high`, `medium-high`, `medium`, `low`), `contact_verified`, and `contact_risk`.
4. **Exact DENUE Headcount Midpoints:** Standardized conversion of employee strata into continuous headcounts (`0 a 5`: 3, `6 a 10`: 8, `11 a 30`: 20, `31 a 50`: 40, `51 a 100`: 75, `101 a 250`: 175, `251 y más`: 350).
5. **Academic Buyer Segmentation:** Classifies workforce into `high_automation_engineering`, `corporate_professional`, `specialized_technical`, or `traditional_labor`.
6. **Persisted 8-File Table Pack:** Persists complete audit, gap, rollup, and KPI tables in `data/processed/tables/` alongside `entities_resolved.parquet` and the interactive Folium map.

In [ ]:
# Cell 2.1: Configuration, Load Stage-01 Enriched Datasets, Map Strata Midpoints & Render Table 1
import os
from pathlib import Path
import polars as pl
from IPython.display import HTML, display

# 1. Centralized Stage-02 Configuration Parameters
STAGE2_CONFIG = {
    "osm_match_radius_m": 50.0,
    "fuzzy_threshold": 0.85,
    "partial_upgrade_score_threshold": 4.8,
    "min_headcount_corporate_partial": 31,
    "max_scrape_workers": 8,
    "center_lat": 20.95379,
    "center_lon": -101.41605,
}

# 2. Resolve file paths across container and host environments
def find_input_file(filename: str) -> Path:
    candidates = [
        Path(f"data/processed/{filename}"),
        Path(f"../data/processed/{filename}"),
        Path(f"notebooks/data/processed/{filename}"),
        Path(f"data/raw/{filename}"),
        Path(f"fixtures/{filename}"),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Cannot locate input file: {filename}")

denue_path = find_input_file("denue_state_11_radius_5km.parquet")
df_denue = pl.read_parquet(denue_path)
print(f" Loaded Stage-01 scored DENUE dataset: {denue_path} ({len(df_denue):,} records, {len(df_denue.columns)} cols)")

gold_path = find_input_file("b2b_gold_list_5d_morning.csv")
df_gold_stage1 = pl.read_csv(gold_path)
print(f" Loaded Stage-01 actionable gold list: {gold_path} ({len(df_gold_stage1):,} email-only accounts)")

try:
    osm_path = find_input_file("osm_amenities.parquet")
    df_osm = pl.read_parquet(osm_path)
    print(f" Loaded ODbL OSM amenities layer: {osm_path} ({len(df_osm):,} POIs)")
except FileNotFoundError:
    df_osm = pl.DataFrame({"osm_id": [], "lat": [], "lon": [], "name": [], "amenity": [], "office": [], "shop": []})
    print("⚠️ OSM layer not found; initializing empty geometry context.")

# 3. Map Employee Strata to Realistic Headcount Midpoints
STRATA_MIDPOINT = {
    "0 a 5 personas": 3,
    "6 a 10 personas": 8,
    "11 a 30 personas": 20,
    "31 a 50 personas": 40,
    "51 a 100 personas": 75,
    "101 a 250 personas": 175,
    "251 y más personas": 350,
}

def map_employee_strata_midpoint(strata: str | None) -> int:
    if not strata or not isinstance(strata, str):
        return 0
    clean = strata.strip().lower()
    for k, v in STRATA_MIDPOINT.items():
        if k in clean or clean in k:
            return v
    if "251" in clean: return 350
    if "101" in clean: return 175
    if "51" in clean: return 75
    if "31" in clean: return 40
    if "11" in clean: return 20
    if "6" in clean: return 8
    if "0" in clean or "1" in clean: return 3
    return 0

df_denue = df_denue.with_columns([
    pl.col("employee_strata").map_elements(map_employee_strata_midpoint, return_dtype=pl.Int64).alias("est_employees_midpoint")
])

# 4. Compute Segment Breakdown & Contact Coverage Matrix
cohorts = ["target", "partial", "no_target"]
table_rows = []
total_recs = len(df_denue)
gap_audit_data = []

for c in cohorts:
    sub = df_denue.filter(pl.col("b2b_status") == c)
    c_total = len(sub)
    em_count = len(sub.filter((pl.col("email").is_not_null()) & (pl.col("email") != "")))
    web_count = len(sub.filter((pl.col("website").is_not_null()) & (pl.col("website") != "")))
    ph_count = len(sub.filter((pl.col("phone").is_not_null()) & (pl.col("phone") != "")))
    stage1_act = len(df_gold_stage1.filter(pl.col("b2b_status") == c)) if "b2b_status" in df_gold_stage1.columns else 0
    gap = c_total - stage1_act if c in ["target", "partial"] else 0
    
    gap_audit_data.append({
        "b2b_segment": c,
        "total_records": c_total,
        "email_count": em_count,
        "website_count": web_count,
        "phone_count": ph_count,
        "stage1_actionable": stage1_act,
        "stage2_opportunity_gap": gap,
    })
    
    badge_color = "#166534" if c == "target" else ("#854d0e" if c == "partial" else "#475569")
    table_rows.append(f"""
    <tr style="border-bottom: 1px solid #e2e8f0;">
        <td style="padding: 8px 12px; font-weight: bold;"><span style="background-color:{badge_color}; color:white; padding:2px 8px; border-radius:4px; font-size:11px;">{c.upper()}</span></td>
        <td style="padding: 8px 12px; text-align: right; font-weight: 600;">{c_total:,}</td>
        <td style="padding: 8px 12px; text-align: right;">{em_count:,} ({em_count/c_total*100:.1f}%)</td>
        <td style="padding: 8px 12px; text-align: right;">{web_count:,} ({web_count/c_total*100:.1f}%)</td>
        <td style="padding: 8px 12px; text-align: right;">{ph_count:,} ({ph_count/c_total*100:.1f}%)</td>
        <td style="padding: 8px 12px; text-align: right; color: #0284c7; font-weight: bold;">{stage1_act:,}</td>
        <td style="padding: 8px 12px; text-align: right; color: #dc2626; font-weight: bold;">+{gap:,} por rescatar</td>
    </tr>
    """)

# 5. Persist Table 1: stage2_contact_gap_audit.csv
df_gap_audit = pl.DataFrame(gap_audit_data)
for t_dir in [Path("data/processed/tables"), Path("../data/processed/tables"), Path("notebooks/data/processed/tables")]:
    if t_dir.parent.exists():
        t_dir.mkdir(parents=True, exist_ok=True)
        df_gap_audit.write_csv(t_dir / "stage2_contact_gap_audit.csv")

html_table_1 = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin: 15px 0;">
    <h4 style="margin: 0 0 8px 0; color: #0f172a;">📊 Tabla 1: Diagnóstico de Entrada Stage-01 y Brecha de Contactabilidad No Explotada</h4>
    <table style="width: 100%; border-collapse: collapse; text-align: left; font-size: 12px; background: white; border: 1px solid #cbd5e1; border-radius: 6px; overflow: hidden;">
        <thead style="background: #0f172a; color: white;">
            <tr>
                <th style="padding: 10px 12px;">Segmento B2B</th>
                <th style="padding: 10px 12px; text-align: right;">Total Registros</th>
                <th style="padding: 10px 12px; text-align: right;">Email DENUE</th>
                <th style="padding: 10px 12px; text-align: right;">Website DENUE</th>
                <th style="padding: 10px 12px; text-align: right;">Teléfono Directo</th>
                <th style="padding: 10px 12px; text-align: right;">Accionables Stage 01</th>
                <th style="padding: 10px 12px; text-align: right;">Brecha Accionable Stage 02</th>
            </tr>
        </thead>
        <tbody>
            {''.join(table_rows)}
            <tr style="background: #f8fafc; font-weight: bold; border-top: 2px solid #94a3b8;">
                <td style="padding: 10px 12px;">TOTAL UNIVERSO</td>
                <td style="padding: 10px 12px; text-align: right;">{total_recs:,}</td>
                <td style="padding: 10px 12px; text-align: right;">{len(df_denue.filter((pl.col('email').is_not_null()) & (pl.col('email') != ''))):,}</td>
                <td style="padding: 10px 12px; text-align: right;">{len(df_denue.filter((pl.col('website').is_not_null()) & (pl.col('website') != ''))):,}</td>
                <td style="padding: 10px 12px; text-align: right;">{len(df_denue.filter((pl.col('phone').is_not_null()) & (pl.col('phone') != ''))):,}</td>
                <td style="padding: 10px 12px; text-align: right; color: #0284c7;">{len(df_gold_stage1):,}</td>
                <td style="padding: 10px 12px; text-align: right; color: #dc2626;">+{total_recs - len(df_gold_stage1):,} candidatos</td>
            </tr>
        </tbody>
    </table>
    <p style="font-size: 11px; color: #64748b; margin-top: 5px;"><i>Nota: Stage 01 filtró estrictamente por email directo existente (49 cuentas). Stage 02 rescata los 32 targets restantes y prospectos Tier-1 parciales mediante herencia corporativa, scraping web y prospección multi-canal.</i></p>
</div>
"""
display(HTML(html_table_1))


## 2. Cross-Source Spatial Resolution (DENUE ↔ OpenStreetMap)

Performs a localized Haversine distance spatial join between DENUE establishment coordinates and OpenStreetMap POIs and building centroids. A match within $\le 50\text{ m}$ confirms active physical storefront or manufacturing operational status while preserving horizontal separation to respect ODbL license boundaries. Generates machine-readable `stage2_osm_verification_audit.csv`.

In [ ]:
# MIGRATES TO: src/mx_osint/resolution/spatial_blocker.py::spatial_join_osm_context
import math
from pathlib import Path
import polars as pl
from IPython.display import HTML, display

def spatial_join_osm_context(denue_df: pl.DataFrame, osm_df: pl.DataFrame, max_dist_m: float = 50.0) -> pl.DataFrame:
    """Cross-source spatial resolution between DENUE and OSM amenities respecting ODbL boundaries."""
    if len(osm_df) == 0:
        return denue_df.with_columns([
            pl.lit(None).cast(pl.Utf8).alias("nearest_osm_id"),
            pl.lit(None).cast(pl.Float64).alias("osm_match_dist_m"),
            pl.lit(False).alias("osm_verified_storefront"),
            pl.lit("").alias("osm_poi_context"),
        ])

    osm_records = osm_df.select(["osm_id", "lat", "lon", "name", "amenity", "office", "shop"]).to_dicts()
    r_earth_m = 6371000.0
    rad = math.pi / 180.0

    nearest_ids = []
    nearest_dists = []
    verified_flags = []
    contexts = []

    for row in denue_df.iter_rows(named=True):
        d_lat, d_lon = row["lat"], row["lon"]
        best_dist = float("inf")
        best_osm = None

        for osm in osm_records:
            o_lat, o_lon = osm["lat"], osm["lon"]
            dlat = (o_lat - d_lat) * rad
            dlon = (o_lon - d_lon) * rad
            a = math.sin(dlat / 2.0) ** 2 + math.cos(d_lat * rad) * math.cos(o_lat * rad) * math.sin(dlon / 2.0) ** 2
            dist_m = 2.0 * math.asin(math.sqrt(a)) * r_earth_m

            if dist_m < best_dist:
                best_dist = dist_m
                best_osm = osm

        if best_osm and best_dist <= 250.0:
            nearest_ids.append(str(best_osm["osm_id"]))
            nearest_dists.append(round(best_dist, 1))
            verified_flags.append(best_dist <= max_dist_m)
            ctx = best_osm.get("name") or best_osm.get("amenity") or best_osm.get("office") or best_osm.get("shop") or ""
            contexts.append(str(ctx))
        else:
            nearest_ids.append(None)
            nearest_dists.append(None)
            verified_flags.append(False)
            contexts.append("")

    return denue_df.with_columns([
        pl.Series("nearest_osm_id", nearest_ids, dtype=pl.Utf8),
        pl.Series("osm_match_dist_m", nearest_dists, dtype=pl.Float64),
        pl.Series("osm_verified_storefront", verified_flags, dtype=pl.Boolean),
        pl.Series("osm_poi_context", contexts, dtype=pl.Utf8),
    ])

df_spatial = spatial_join_osm_context(df_denue, df_osm, max_dist_m=STAGE2_CONFIG["osm_match_radius_m"])
verified_count = len(df_spatial.filter(pl.col("osm_verified_storefront") == True))
print(f"✅ Spatial join complete: {verified_count:,} de {len(df_spatial):,} establecimientos verificados físicamente (≤50m de POI/inmueble OSM)")

# 3. Persist Table 2 Audit: stage2_osm_verification_audit.csv
osm_audit_cols = ["row_id", "denue_id", "business_name", "legal_name", "lat", "lon", "b2b_status", "b2b_5d_score", "osm_verified_storefront", "osm_match_dist_m", "nearest_osm_id", "osm_poi_context"]
df_osm_audit = df_spatial.select([c for c in osm_audit_cols if c in df_spatial.columns])
for t_dir in [Path("data/processed/tables"), Path("../data/processed/tables"), Path("notebooks/data/processed/tables")]:
    if t_dir.parent.exists():
        t_dir.mkdir(parents=True, exist_ok=True)
        df_osm_audit.write_csv(t_dir / "stage2_osm_verification_audit.csv")

# 4. Render Table 2: Top 15 OSM Spatial Matches
sample_osm = (
    df_spatial.filter(pl.col("osm_verified_storefront") == True)
    .sort("osm_match_dist_m")
    .head(15)
)

t2_rows = []
for r in sample_osm.iter_rows(named=True):
    t2_rows.append(f"""
    <tr style="border-bottom: 1px solid #e2e8f0;">
        <td style="padding: 6px 10px; font-family: monospace; font-size: 11px;">{r.get('denue_id', r.get('row_id'))}</td>
        <td style="padding: 6px 10px; font-weight: 600;">{r.get('business_name', '')}</td>
        <td style="padding: 6px 10px; color: #475569;">{r.get('legal_name', '') or '-'}</td>
        <td style="padding: 6px 10px; text-align: center;"><span style="background:#dcfce7; color:#15803d; padding:2px 6px; border-radius:3px; font-size:11px; font-weight:bold;">✅ {r.get('osm_match_dist_m')} m</span></td>
        <td style="padding: 6px 10px; font-family: monospace; font-size: 11px; color:#0369a1;">{r.get('nearest_osm_id')}</td>
        <td style="padding: 6px 10px; color: #334155;"><b>{r.get('osm_poi_context')}</b></td>
    </tr>
    """)

html_table_2 = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin: 15px 0;">
    <h4 style="margin: 0 0 8px 0; color: #0f172a;">🗺️ Tabla 2: Resolución Espacial Cruzada DENUE ↔ OSM (Muestra de Fachadas Verificadas ≤50m)</h4>
    <table style="width: 100%; border-collapse: collapse; text-align: left; font-size: 12px; background: white; border: 1px solid #cbd5e1; border-radius: 6px; overflow: hidden;">
        <thead style="background: #1e293b; color: white;">
            <tr>
                <th style="padding: 8px 10px;">ID DENUE</th>
                <th style="padding: 8px 10px;">Nombre Comercial</th>
                <th style="padding: 8px 10px;">Razón Social</th>
                <th style="padding: 8px 10px; text-align: center;">Distancia OSM</th>
                <th style="padding: 8px 10px;">ID OSM (ODbL)</th>
                <th style="padding: 8px 10px;">Contexto POI / Inmueble</th>
            </tr>
        </thead>
        <tbody>
            {''.join(t2_rows)}
        </tbody>
    </table>
    <p style="font-size: 11px; color: #64748b; margin-top: 5px;"><i>Nota: La capa OSM valida la existencia física del establecimiento. Guardado en stage2_osm_verification_audit.csv.</i></p>
</div>
"""
display(HTML(html_table_2))


## 3. NLP Corporate Campus Deduplication, Contact Inheritance & Academic Workforce Profiling

Normalizes Mexican legal names (`S.A. DE C.V.`, `S. DE R.L.`, `S.A.P.I.`), strips corporate branch qualifiers (`PLANTA`, `MATRIZ`, `SUCURSAL`), clusters multi-plant campuses with pairwise disambiguation (zero false merges on generic words), propagates contacts from headquarters to secondary facilities, enforces mathematical canonical reconciliation, and segments workforce academic buyer tiers (`high_automation_engineering`, `corporate_professional`, `specialized_technical`, `traditional_labor`).

In [ ]:
# MIGRATES TO: src/mx_osint/resolution/mex_name_normalizer.py::cluster_corporate_entities
import hashlib
from pathlib import Path
import re
import unicodedata
from collections import defaultdict
import polars as pl
from rapidfuzz import fuzz
from IPython.display import HTML, display

LEGAL_SUFFIX_REGEX = re.compile(
    r"\b("
    r"S\.?\s*A\.?\s*P\.?\s*I\.?\s*(?:DE\s*C\.?\s*V\.?)?|"
    r"S\.?\s*A\.?\s*B\.?\s*(?:DE\s*C\.?\s*V\.?)?|"
    r"S\.?\s*A\.?\s*DE\s*C\.?\s*V\.?|"
    r"S\.?\s*DE\s*R\.?\s*L\.?\s*(?:DE\s*C\.?\s*V\.?)?|"
    r"S\.?\s*C\.?\s*DE\s*R\.?\s*L\.?|"
    r"S\.?\s*C\.?|"
    r"A\.?\s*C\.?|"
    r"S\.?\s*A\.?|"
    r"SAB\s*DE\s*CV"
    r")\b",
    re.IGNORECASE
)

BRANCH_QUALIFIER_REGEX = re.compile(
    r"\b(PLANTA|MATRIZ|SUCURSAL|CORPORATIVO|CAMPUS|BODEGA|NORTE|SUR|LEON|GUANAJUATO|BAJIO|MEXICO)\b",
    re.IGNORECASE
)

PUNCTUATION_REGEX = re.compile(r"[,.\-'\"]")

def normalize_mexican_legal_name(name: str) -> str:
    if not name or not isinstance(name, str): return ""
    text = unicodedata.normalize("NFKD", name)
    text = "".join(c for c in text if not unicodedata.combining(c)).upper()
    text = LEGAL_SUFFIX_REGEX.sub(" ", text)
    text = PUNCTUATION_REGEX.sub(" ", text)
    return " ".join(text.split()).strip()

def strip_corporate_branch_qualifiers(name: str) -> str:
    cleaned = normalize_mexican_legal_name(name)
    cleaned = BRANCH_QUALIFIER_REGEX.sub(" ", cleaned)
    cleaned = PUNCTUATION_REGEX.sub(" ", cleaned)
    return " ".join(cleaned.split()).strip()

def classify_workforce_academic_level(wage_mxn: float | None, degree_share: float | None) -> str:
    wage = float(wage_mxn or 0.0)
    share = float(degree_share or 0.0)
    if share >= 0.60 and wage >= 25000:
        return "high_automation_engineering"
    elif share >= 0.35 and wage >= 18000:
        return "corporate_professional"
    elif share >= 0.20 or wage >= 15000:
        return "specialized_technical"
    else:
        return "traditional_labor"

def cluster_corporate_entities(df: pl.DataFrame, fuzzy_threshold: float = 0.85) -> pl.DataFrame:
    name_col = "business_name" if "business_name" in df.columns else "nombre_estab"
    legal_col = "legal_name" if "legal_name" in df.columns else "raz_social"
    
    norm_commercial = [normalize_mexican_legal_name(str(n or '')) for n in df[name_col].to_list()]
    parent_commercial = [strip_corporate_branch_qualifiers(str(n or '')) for n in df[name_col].to_list()]
    norm_legal = [normalize_mexican_legal_name(str(l or '')) if l else '' for l in df[legal_col].to_list()]
    parent_legal = [strip_corporate_branch_qualifiers(str(l or '')) if l else '' for l in df[legal_col].to_list()]

    df_prep = df.with_columns([
        pl.Series("norm_name", norm_commercial),
        pl.Series("parent_name", parent_commercial),
        pl.Series("norm_legal", norm_legal),
        pl.Series("parent_legal", parent_legal),
    ])

    GENERIC_STOPWORDS = {'SERVICIOS', 'COMERCIAL', 'GRUPO', 'DISTRIBUIDORA', 'LOGISTICA', 'CENTRO', 'DESPACHO', 'TIENDA', 'ABARROTES', 'TALLER', 'FARMACIA', 'CORPORATIVO', 'CONSULTORIA', 'SOLUCIONES', 'TRANSPORTES', 'INMOBILIARIA', 'DESARROLLO', 'TECNOLOGIA'}
    records = df_prep.to_dicts()
    n = len(records)
    parent_map = list(range(n))

    def find(i):
        if parent_map[i] == i: return i
        parent_map[i] = find(parent_map[i])
        return parent_map[i]

    def union(i, j):
        ri, rj = find(i), find(j)
        if ri != rj: parent_map[rj] = ri

    FREE_DOMAINS = {"gmail.com", "hotmail.com", "outlook.com", "yahoo.com", "live.com", "live.com.mx", "prodigy.net.mx", "gob.mx"}

    for i in range(n):
        ri = records[i]
        legi = (ri.get("parent_legal") or ri.get(legal_col) or "").strip().upper()
        pcommi = (ri.get("parent_name") or ri.get(name_col) or "").strip().upper()
        emi = (ri.get("email") or "").strip().lower()
        domi = emi.split("@")[1] if ("@" in emi and emi.split("@")[1] not in FREE_DOMAINS) else ""

        for j in range(i + 1, n):
            rj = records[j]
            legj = (rj.get("parent_legal") or rj.get(legal_col) or "").strip().upper()
            pcommj = (rj.get("parent_name") or rj.get(name_col) or "").strip().upper()
            emj = (rj.get("email") or "").strip().lower()
            domj = emj.split("@")[1] if ("@" in emj and emj.split("@")[1] not in FREE_DOMAINS) else ""

            match = False
            # 1. Exact parent legal or parent commercial name match
            if legi and legj and legi == legj and len(legi) >= 5 and legi not in GENERIC_STOPWORDS:
                match = True
            elif pcommi and pcommj and pcommi == pcommj and len(pcommi) >= 5 and pcommi not in GENERIC_STOPWORDS:
                match = True
            # 2. Shared corporate email or corporate domain (merges Lear, Bodycote, Hotsson/Milenium)
            elif emi and emj and emi == emj:
                match = True
            elif domi and domj and domi == domj and not any(k in domi for k in ["edu.mx", "org.mx"]):
                match = True
            # 3. High-confidence token sort ratio match on stripped parent legal names (>= 92%)
            elif legi and legj and len(legi) >= 8 and len(legj) >= 8 and legi not in GENERIC_STOPWORDS and legj not in GENERIC_STOPWORDS:
                if fuzz.token_sort_ratio(legi, legj) >= 92:
                    match = True

            if match:
                union(i, j)

    clusters = defaultdict(list)
    for i in range(n):
        clusters[find(i)].append(i)

    canonical_ids = [None] * n
    canonical_names = [None] * n
    is_duplicate = [False] * n
    inherited_emails = [None] * n
    inherited_websites = [None] * n
    inherited_phones = [None] * n
    contact_inheritance_scores = ["none"] * n

    for root, indices in clusters.items():
        def priority(idx):
            r = records[idx]
            name = str(r.get(name_col, '')).upper()
            is_mat = 1 if 'MATRIZ' in name or 'CORPORATIVO' in name else 0
            has_em = 1 if r.get('email') else 0
            score = float(r.get('b2b_5d_score') or 0.0)
            return (is_mat, has_em, score)

        sorted_idx = sorted(indices, key=priority, reverse=True)
        prim_idx = sorted_idx[0]
        prim_rec = records[prim_idx]

        leg_clean = normalize_mexican_legal_name(prim_rec.get(legal_col) or '')
        canon_name = leg_clean if (leg_clean and len(leg_clean) >= 5 and leg_clean not in GENERIC_STOPWORDS) else (prim_rec.get('parent_name') or prim_rec.get(name_col))
        canon_id = f"ent_{hashlib.sha1(canon_name.encode('utf-8')).hexdigest()[:12]}"

        best_em = next((str(records[idx]['email']).strip() for idx in sorted_idx if records[idx].get('email')), '')
        best_web = next((str(records[idx]['website']).strip() for idx in sorted_idx if records[idx].get('website')), '')
        best_ph = next((str(records[idx]['phone']).strip() for idx in sorted_idx if records[idx].get('phone')), '')

        for rank, idx in enumerate(sorted_idx):
            canonical_ids[idx] = canon_id
            canonical_names[idx] = canon_name
            is_duplicate[idx] = (rank > 0)

            curr_em = str(records[idx].get('email') or '').strip()
            curr_web = str(records[idx].get('website') or '').strip()
            curr_ph = str(records[idx].get('phone') or '').strip()
            
            inherited_emails[idx] = best_em if (not curr_em and best_em) else curr_em
            inherited_websites[idx] = best_web if (not curr_web and best_web) else curr_web
            inherited_phones[idx] = best_ph if (not curr_ph and best_ph) else curr_ph
            
            if not curr_em and best_em:
                contact_inheritance_scores[idx] = "high" if "MATRIZ" in str(prim_rec.get(name_col, '')).upper() else "medium"
            elif curr_em:
                contact_inheritance_scores[idx] = "direct"

    df_out = df_prep.with_columns([
        pl.Series("canonical_entity_id", canonical_ids),
        pl.Series("canonical_parent_name", canonical_names),
        pl.Series("is_duplicate", is_duplicate),
        pl.Series("inherited_email", inherited_emails),
        pl.Series("inherited_website", inherited_websites),
        pl.Series("inherited_phone", inherited_phones),
        pl.Series("contact_inheritance_score", contact_inheritance_scores),
    ])

    counts = df_out.group_by("canonical_entity_id").agg([
        pl.len().alias("campus_site_count"),
        pl.col("est_employees_midpoint").sum().alias("total_est_employees_midpoint"),
        pl.col("est_monthly_payroll_mxn").sum().alias("campus_consolidated_payroll_mxn"),
    ])
    return df_out.join(counts, on="canonical_entity_id", how="left")

df_clustered = cluster_corporate_entities(df_spatial, fuzzy_threshold=STAGE2_CONFIG["fuzzy_threshold"])

# 5. Mathematical Reconciliation Verification (Invariance Check)
total_records = len(df_clustered)
duplicate_records = len(df_clustered.filter(pl.col("is_duplicate") == True))
unique_canonical = df_clustered["canonical_entity_id"].n_unique()
reconciliation_ok = (total_records - duplicate_records == unique_canonical)
assert reconciliation_ok, f"Reconciliation mismatch: {total_records} - {duplicate_records} != {unique_canonical}"
print(f"✅ Mathematical reconciliation passed: {total_records} total - {duplicate_records} duplicates = {unique_canonical} unique canonical accounts")

# 6. Workforce Academic Level Classification
academic_tiers = [
    classify_workforce_academic_level(w, s)
    for w, s in zip(df_clustered["est_monthly_wage_mxn"].to_list(), df_clustered["est_degree_share"].to_list())
]
df_clustered = df_clustered.with_columns(pl.Series("workforce_academic_level", academic_tiers))

# 7. Persist Table 3 & Academic Rollup Tables
campus_summary = (
    df_clustered.filter((pl.col("campus_site_count") > 1) & (pl.col("b2b_status") != "no_target"))
    .group_by(["canonical_entity_id", "canonical_parent_name"])
    .agg([
        pl.len().alias("sites"),
        pl.col("business_name").str.join(" | ").alias("plantas"),
        pl.col("total_est_employees_midpoint").first().alias("total_headcount"),
        pl.col("campus_consolidated_payroll_mxn").first().alias("nomina_total"),
        pl.col("inherited_email").first().alias("email_campus"),
        pl.col("inherited_website").first().alias("web_campus"),
    ])
    .sort("nomina_total", descending=True)
)

academic_summary = df_clustered.select([
    "row_id", "denue_id", "business_name", "canonical_entity_id", "canonical_parent_name",
    "est_monthly_wage_mxn", "est_degree_share", "est_employees_midpoint", "workforce_academic_level", "b2b_5d_score"
])

for t_dir in [Path("data/processed/tables"), Path("../data/processed/tables"), Path("notebooks/data/processed/tables")]:
    if t_dir.parent.exists():
        t_dir.mkdir(parents=True, exist_ok=True)
        campus_summary.write_csv(t_dir / "stage2_campus_rollup.csv")
        academic_summary.write_csv(t_dir / "stage2_workforce_academic_density.csv")

# 8. Render Table 3: Consolidated Multi-Plant Corporate Campuses
t3_rows = []
for r in campus_summary.head(15).iter_rows(named=True):
    em_tag = f"<span style='color:#16a34a; font-weight:bold;'>{r['email_campus']}</span>" if r['email_campus'] else "<span style='color:#94a3b8;'>Sin email</span>"
    t3_rows.append(f"""
    <tr style="border-bottom: 1px solid #e2e8f0;">
        <td style="padding: 8px 10px; font-weight: bold; color: #1e3a8a;">{r['canonical_parent_name']}</td>
        <td style="padding: 8px 10px; text-align: center;"><span style="background:#e0e7ff; color:#3730a3; padding:2px 8px; border-radius:10px; font-weight:bold;">{r['sites']} sitios</span></td>
        <td style="padding: 8px 10px; text-align: right; font-weight: 600;">{r['total_headcount']:,} pers.</td>
        <td style="padding: 8px 10px; text-align: right; font-weight: bold; color:#0f766e;">${r['nomina_total']/1e6:.2f}M MXN</td>
        <td style="padding: 8px 10px; font-size: 11px;">{em_tag}</td>
    </tr>
    """)

html_table_3 = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin: 15px 0;">
    <h4 style="margin: 0 0 8px 0; color: #0f172a;">🏛️ Tabla 3: Grupos Corporativos y Campus Industriales Consolidados (Top 15 por Nómina)</h4>
    <table style="width: 100%; border-collapse: collapse; text-align: left; font-size: 12px; background: white; border: 1px solid #cbd5e1; border-radius: 6px; overflow: hidden;">
        <thead style="background: #1e1b4b; color: white;">
            <tr>
                <th style="padding: 8px 10px;">Entidad Matriz Canónica</th>
                <th style="padding: 8px 10px; text-align: center;">Sitios</th>
                <th style="padding: 8px 10px; text-align: right;">Plantilla Estimada</th>
                <th style="padding: 8px 10px; text-align: right;">Nómina Total Estimada</th>
                <th style="padding: 8px 10px;">Contacto Heredado</th>
            </tr>
        </thead>
        <tbody>
            {''.join(t3_rows)}
        </tbody>
    </table>
    <p style="font-size: 11px; color: #64748b; margin-top: 5px;"><i>Nota: Persistido en stage2_campus_rollup.csv y stage2_workforce_academic_density.csv.</i></p>
</div>
"""
display(HTML(html_table_3))

## 4. Deep Contact Enrichment, Multi-Channel Rescue & Confidence Scoring

Expands web scraping eligibility across all priority candidates (`target`, `is_rescued_partial`, and corporate partials with score $\ge 4.8$). Classifies emails and channels into an explicit **Contact Confidence & Provenance Hierarchy**:
- `high`: Direct verified email in DENUE
- `medium-high`: Directly scraped visible website email
- `medium`: Campus-inherited corporate email from primary plant
- `low` / `unverified`: Synthesized role inboxes (`contacto@`, `rh@`, `compras@`) — flagged with `contact_risk: high` so they are not included in automated sequence campaigns without manual review.

In [ ]:
# MIGRATES TO: src/mx_osint/resolution/contact_enricher.py::enrich_and_rescue_b2b_contacts
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import re
from urllib.parse import quote, urljoin
import polars as pl
import requests
from IPython.display import HTML, display

EMAIL_REGEX = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")

def classify_b2b_email_role(email: str | None) -> str:
    if not email or not isinstance(email, str) or "@" not in email:
        return "none"
    clean_em = email.strip().lower()
    free_domains = ["gmail.com", "hotmail.com", "outlook.com", "yahoo.com", "live.com", "prodigy.net.mx"]
    if any(clean_em.endswith(f"@{d}") for d in free_domains): return "free_personal"
    user_part = clean_em.split("@")[0]
    decision_makers = ["director", "gerente", "rh", "recursoshumanos", "purchasing", "compras", "talento", "ceo", "cfo", "planta"]
    generic_roles = ["contacto", "info", "recepcion", "atencion", "ventas", "servicio", "facturacion", "admin"]
    if any(dm in user_part for dm in decision_makers): return "decision_maker_role"
    if any(gen in user_part for gen in generic_roles): return "generic_role_inbox"
    return "corporate_direct"

def synthesize_corporate_domain_contact(website: str | None) -> dict:
    if not website or not isinstance(website, str):
        return {"domain": "", "synthesized_email": "", "alternate_inboxes": []}
    clean_w = website.strip().lower()
    for pfx in ["https://", "http://", "www."]:
        if clean_w.startswith(pfx): clean_w = clean_w[len(pfx):]
    clean_w = clean_w.split("/")[0].split("?")[0].strip()
    non_corp = ["facebook.com", "instagram.com", "twitter.com", "linkedin.com", "google.com", "maps.google.com", "gob.mx", "wixsite.com"]
    if not clean_w or "." not in clean_w or any(clean_w.endswith(fd) for fd in non_corp):
        return {"domain": "", "synthesized_email": "", "alternate_inboxes": []}
    return {
        "domain": clean_w,
        "synthesized_email": f"contacto@{clean_w}",
        "alternate_inboxes": [f"rh@{clean_w}", f"compras@{clean_w}", f"ventas@{clean_w}"],
    }

def scrape_corporate_contact_emails(websites: list[str], max_workers: int = 8, timeout: float = 2.5) -> dict[str, str]:
    scraped = {}
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) MexicoCommercialOSINT/1.0"}
    clean_targets = list(set(w.strip() for w in websites if w and isinstance(w, str) and "." in w))
    
    def scrape_url(w: str):
        url = w if w.lower().startswith("http") else f"https://{w}"
        for target in [url, urljoin(url, "/contacto"), urljoin(url, "/contact")]:
            try:
                resp = requests.get(target, headers=headers, timeout=timeout)
                if resp.status_code == 200:
                    found = EMAIL_REGEX.findall(resp.text)
                    valid = [
                        e for e in found 
                        if not any(e.lower().endswith(ext) for ext in [".png", ".jpg", ".jpeg", ".gif", ".svg", ".webp", ".js", ".css"])
                        and not any(e.lower().startswith(p) for p in ["bootstrap@", "jquery@", "user@"])
                        and len(e) <= 50
                    ]
                    if valid:
                        return (w, valid[0])
            except Exception:
                pass
        return (w, "")

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(scrape_url, w) for w in clean_targets]
        for f in as_completed(futures):
            w_res, em_res = f.result()
            if em_res:
                scraped[w_res] = em_res
    return scraped

def evaluate_stage2_actionability(rec: dict, scraped_map: dict | None = None) -> dict:
    b2b_status = rec.get("b2b_status", "")
    is_rescued = rec.get("is_rescued_partial", False)
    email = (rec.get("email") or "").strip()
    inherited_em = (rec.get("inherited_email") or "").strip()
    website = (rec.get("website") or "").strip()
    phone = (rec.get("phone") or "").strip()
    score = float(rec.get("b2b_5d_score") or 0.0)
    headcount = int(rec.get("est_employees_midpoint") or 0)
    synth = synthesize_corporate_domain_contact(website)

    # 1. Direct Email (Target or Stage-01 Rescued Partial)
    if (b2b_status == "target" or is_rescued) and email:
        return {
            "is_stage2_actionable": True,
            "actionable_channel": "Direct Email (Verified DENUE)",
            "rescue_reason": "Stage-01 Verified Direct Email",
            "effective_contact": email,
            "email_source": "direct_denue",
            "email_confidence": "high",
            "contact_verified": True,
            "contact_risk": "low",
            "actionable_tier": "Tier-1 Direct Email",
        }

    # 2. Campus Inherited Email
    if (b2b_status == "target" or is_rescued) and inherited_em:
        return {
            "is_stage2_actionable": True,
            "actionable_channel": "Campus Inherited Email (Parent Entity)",
            "rescue_reason": "Campus Contact Inheritance",
            "effective_contact": inherited_em,
            "email_source": "campus_inherited",
            "email_confidence": "medium",
            "contact_verified": False,
            "contact_risk": "medium",
            "actionable_tier": "Tier-1 Campus Inherited",
        }

    # 2.5. Direct Email Scraped from Corporate Website
    scraped_em = (scraped_map or {}).get(website, "")
    if (b2b_status == "target" or is_rescued) and scraped_em:
        return {
            "is_stage2_actionable": True,
            "actionable_channel": "Direct Email (Scraped Corporate Website)",
            "rescue_reason": "Live Corporate Website Email Scraping",
            "effective_contact": scraped_em,
            "email_source": "scraped_website",
            "email_confidence": "medium-high",
            "contact_verified": True,
            "contact_risk": "low",
            "actionable_tier": "Tier-1 Direct Email",
        }

    # 3. Target without email, but has corporate website -> Domain Synthesized
    if (b2b_status == "target" or is_rescued) and synth["synthesized_email"]:
        return {
            "is_stage2_actionable": True,
            "actionable_channel": "Website / Synthesized Domain Email",
            "rescue_reason": "Corporate Web Domain Contact Synthesis",
            "effective_contact": synth["synthesized_email"],
            "email_source": "synthesized_domain",
            "email_confidence": "low",
            "contact_verified": False,
            "contact_risk": "high",
            "actionable_tier": "Tier-2 Domain Synthesized",
        }

    # 4. Tier-1 Corporate Domain Partial Upgrade
    if b2b_status == "partial" and score >= STAGE2_CONFIG["partial_upgrade_score_threshold"] and email:
        return {
            "is_stage2_actionable": True,
            "actionable_channel": "Direct Email (Rescued Tier-1 Corporate)",
            "rescue_reason": "Tier-1 Corporate Domain Partial Upgrade",
            "effective_contact": email,
            "email_source": "direct_denue",
            "email_confidence": "high",
            "contact_verified": True,
            "contact_risk": "low",
            "actionable_tier": "Tier-1 Rescued Partial",
        }
    elif b2b_status == "partial" and score >= STAGE2_CONFIG["partial_upgrade_score_threshold"] and synth["synthesized_email"] and headcount >= STAGE2_CONFIG["min_headcount_corporate_partial"]:
        return {
            "is_stage2_actionable": True,
            "actionable_channel": "Website / Synthesized Domain Email",
            "rescue_reason": "Tier-1 Headcount & Domain Partial Upgrade",
            "effective_contact": synth["synthesized_email"],
            "email_source": "synthesized_domain",
            "email_confidence": "low",
            "contact_verified": False,
            "contact_risk": "high",
            "actionable_tier": "Tier-2 Rescued Partial",
        }

    # 5. Direct Phone + LinkedIn ABM (Target without email/domain)
    if (b2b_status == "target" or is_rescued) and phone:
        return {
            "is_stage2_actionable": True,
            "actionable_channel": "Direct Phone + LinkedIn ABM",
            "rescue_reason": "Multi-Channel Phone & Executive Search Rescue",
            "effective_contact": f"Tel: {phone}",
            "email_source": "none",
            "email_confidence": "none",
            "contact_verified": True,
            "contact_risk": "medium",
            "actionable_tier": "Tier-3 Phone / ABM",
        }

    # 6. Field Sales + LinkedIn ABM (Remaining high-priority targets)
    if b2b_status == "target" or is_rescued:
        return {
            "is_stage2_actionable": True,
            "actionable_channel": "Field Sales + LinkedIn ABM",
            "rescue_reason": "Storefront Physical Visit & Executive Search Rescue",
            "effective_contact": "Visita Presencial / Ruta",
            "email_source": "none",
            "email_confidence": "none",
            "contact_verified": False,
            "contact_risk": "high",
            "actionable_tier": "Tier-4 Field Sales / ABM",
        }

    return {
        "is_stage2_actionable": False,
        "actionable_channel": "Non-Actionable (Low Priority)",
        "rescue_reason": "Does not meet commercial qualification criteria",
        "effective_contact": "-",
        "email_source": "none",
        "email_confidence": "none",
        "contact_verified": False,
        "contact_risk": "low",
        "actionable_tier": "Inactive",
    }

# Apply evaluation across all records
# Live Scrape Candidate Websites
candidate_websites = [
    r.get("website") for r in df_clustered.iter_rows(named=True)
    if (r.get("b2b_status") in ["target", "partial"] or r.get("is_rescued_partial"))
    and not r.get("email") and not r.get("inherited_email") and r.get("website")
]
scraped_contacts = scrape_corporate_contact_emails(candidate_websites, max_workers=STAGE2_CONFIG.get("max_scrape_workers", 8))

# Apply evaluation across all records
eval_results = [evaluate_stage2_actionability(r, scraped_map=scraped_contacts) for r in df_clustered.iter_rows(named=True)]

# Build LinkedIn Search Queries
linkedin_urls = []
for p_name, loc in zip(df_clustered["canonical_parent_name"].to_list(), df_clustered["municipality"].to_list()):
    q = quote(f"{p_name} {loc or 'Guanajuato'}")
    linkedin_urls.append(f"https://www.linkedin.com/search/results/people/?keywords={q}")

df_enriched = df_clustered.with_columns([
    pl.Series("is_stage2_actionable", [e["is_stage2_actionable"] for e in eval_results]),
    pl.Series("actionable_channel", [e["actionable_channel"] for e in eval_results]),
    pl.Series("rescue_reason", [e["rescue_reason"] for e in eval_results]),
    pl.Series("effective_contact", [e["effective_contact"] for e in eval_results]),
    pl.Series("email_source", [e["email_source"] for e in eval_results]),
    pl.Series("email_confidence", [e["email_confidence"] for e in eval_results]),
    pl.Series("contact_verified", [e["contact_verified"] for e in eval_results]),
    pl.Series("contact_risk", [e["contact_risk"] for e in eval_results]),
    pl.Series("actionable_tier", [e["actionable_tier"] for e in eval_results]),
    pl.Series("email_role_tier", [classify_b2b_email_role(em) for em in df_clustered["email"].to_list()]),
    pl.Series("linkedin_search_url", linkedin_urls),
])

# Enforce commercial qualification: no_target is excluded from actionable master
act_stage2_sites = df_enriched.filter((pl.col("is_stage2_actionable") == True) & (pl.col("b2b_status") != "no_target"))
print(f"🚀 Multi-channel expansion complete: {len(act_stage2_sites):,} sitios físicos accionables en Stage 02 vs {len(df_gold_stage1):,} en Stage 01 (+{len(act_stage2_sites)-len(df_gold_stage1):,} sitios, +{(len(act_stage2_sites)/len(df_gold_stage1)-1)*100:.1f}%)")

# 5. Persist Missing Enrichment Gap Table & KPI Table
df_missing_gap = (
    act_stage2_sites.filter(pl.col("email_confidence").is_in(["low", "none"]))
    .select(["denue_id", "business_name", "canonical_parent_name", "b2b_status", "b2b_5d_score", "website", "phone", "actionable_channel", "effective_contact", "email_confidence", "rescue_reason"])
    .with_columns(pl.lit("LinkedIn Executive Outreach / Phone Qualification").alias("next_action"))
)

kpi_data = [{
    "stage1_actionable_accounts": len(df_gold_stage1),
    "stage2_actionable_sites": len(act_stage2_sites),
    "net_actionable_gain": len(act_stage2_sites) - len(df_gold_stage1),
    "growth_pct": (len(act_stage2_sites) / len(df_gold_stage1) - 1) * 100,
    "high_confidence_email_count": len(act_stage2_sites.filter(pl.col("email_confidence") == "high")),
    "medium_confidence_count": len(act_stage2_sites.filter(pl.col("email_confidence") == "medium")),
    "low_confidence_synthetic_count": len(act_stage2_sites.filter(pl.col("email_confidence") == "low")),
}]
df_kpi = pl.DataFrame(kpi_data)

for t_dir in [Path("data/processed/tables"), Path("../data/processed/tables"), Path("notebooks/data/processed/tables")]:
    if t_dir.parent.exists():
        t_dir.mkdir(parents=True, exist_ok=True)
        df_missing_gap.write_csv(t_dir / "stage2_missing_enrichment_gap.csv")
        df_kpi.write_csv(t_dir / "stage2_enrichment_coverage_kpi.csv")

# 6. Render Table 4: Actionable Expansion Funnel & Newly Unlocked Accounts Sample
channel_counts = act_stage2_sites.group_by("actionable_channel").agg(pl.len().alias("total")).sort("total", descending=True)

funnel_rows = []
for ch in channel_counts.iter_rows(named=True):
    funnel_rows.append(f"""
    <tr style="border-bottom: 1px solid #e2e8f0;">
        <td style="padding: 6px 10px; font-weight: 600;">{ch['actionable_channel']}</td>
        <td style="padding: 6px 10px; text-align: right; font-weight: bold; color: #0f766e;">{ch['total']:,}</td>
        <td style="padding: 6px 10px; text-align: right; color: #64748b;">{ch['total']/len(act_stage2_sites)*100:.1f}%</td>
    </tr>
    """)

newly_unlocked = (
    act_stage2_sites.filter(pl.col("rescue_reason") != "Stage-01 Verified Direct Email")
    .sort("b2b_5d_score", descending=True)
    .head(15)
)

unlocked_rows = []
for r in newly_unlocked.iter_rows(named=True):
    risk_color = "#15803d" if r['contact_risk'] == 'low' else ("#b45309" if r['contact_risk'] == 'medium' else "#b91c1c")
    conf_badge = f"<span style='background:#f1f5f9; color:{risk_color}; padding:2px 6px; border-radius:3px; font-weight:bold; font-size:10px;'>{r['email_confidence'].upper()} ({r['contact_risk'].upper()} RISK)</span>"
    unlocked_rows.append(f"""
    <tr style="border-bottom: 1px solid #e2e8f0;">
        <td style="padding: 6px 10px; font-weight: 600;">{r.get('business_name')}</td>
        <td style="padding: 6px 10px; color: #475569;">{r.get('canonical_parent_name')}</td>
        <td style="padding: 6px 10px; text-align: center; font-weight: bold; color:#1d4ed8;">{float(r.get('b2b_5d_score') or 0.0):.2f} pts</td>
        <td style="padding: 6px 10px; font-size: 11px;"><span style="background:#dbeafe; color:#1e40af; padding:2px 6px; border-radius:3px;">{r.get('actionable_channel')}</span></td>
        <td style="padding: 6px 10px; font-family: monospace; font-size: 11px; color:#166534;">{r.get('effective_contact')}</td>
        <td style="padding: 6px 10px; text-align:center;">{conf_badge}</td>
    </tr>
    """)

html_table_4 = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin: 15px 0;">
    <h4 style="margin: 0 0 8px 0; color: #0f172a;">📈 Tabla 4: Embudo de Expansión de Prospectos Accionables (Stage 01 ➔ Stage 02)</h4>
    <div style="display: grid; grid-template-columns: 1fr 2fr; gap: 15px; margin-bottom: 15px;">
        <div style="background: white; border: 1px solid #cbd5e1; border-radius: 6px; padding: 10px;">
            <h5 style="margin: 0 0 6px 0; color: #1e293b;">Desglose por Canal Accionable</h5>
            <table style="width: 100%; border-collapse: collapse; font-size: 11px;">
                <thead><tr style="background:#f1f5f9;"><th style="padding:4px;">Canal</th><th style="padding:4px; text-align:right;">Cuentas</th><th style="padding:4px; text-align:right;">%</th></tr></thead>
                <tbody>{''.join(funnel_rows)}</tbody>
            </table>
        </div>
        <div style="background: #f8fafc; border: 1px solid #cbd5e1; border-radius: 6px; padding: 12px; display: flex; flex-direction: column; justify-content: center;">
            <div style="font-size: 13px; color: #334155; margin-bottom: 8px;"><b>Salto de Cobertura Comercial:</b></div>
            <div style="font-size: 24px; font-weight: bold; color: #0f766e;">{len(df_gold_stage1):,} ➔ {len(act_stage2_sites):,} Sitios Accionables</div>
            <div style="font-size: 12px; color: #64748b; margin-top: 4px;">Desbloqueo de <b>{len(act_stage2_sites)-len(df_gold_stage1):,} sitios de alto valor</b> con jerarquía de confianza (High/Medium/Low) para control de rebotes y reputación de dominio.</div>
        </div>
    </div>
    <h5 style="margin: 10px 0 6px 0; color: #1e293b;">Muestra de Nuevos Prospectos con Score de Confianza de Contacto</h5>
    <table style="width: 100%; border-collapse: collapse; text-align: left; font-size: 11px; background: white; border: 1px solid #cbd5e1; border-radius: 6px; overflow: hidden;">
        <thead style="background: #064e3b; color: white;">
            <tr>
                <th style="padding: 8px 10px;">Establecimiento</th>
                <th style="padding: 8px 10px;">Entidad Matriz</th>
                <th style="padding: 8px 10px; text-align: center;">Score 5D</th>
                <th style="padding: 8px 10px;">Canal Asignado</th>
                <th style="padding: 8px 10px;">Contacto Resuelto</th>
                <th style="padding: 8px 10px; text-align: center;">Confianza / Riesgo</th>
            </tr>
        </thead>
        <tbody>
            {''.join(unlocked_rows)}
        </tbody>
    </table>
</div>
"""
display(HTML(html_table_4))

## 5. Master Production Contracts, Table Pack & Interactive Map

Exports the complete Stage-02 production contract:
1. **`stage2_canonical_action_master.csv` / `.parquet`:** Exactly one row per canonical parent entity (deduplicated accounts for CRM & sales sequences).
2. **`stage2_site_action_list.csv` / `.parquet`:** One row per physical establishment for field routes and logistical planning.
3. **`entities_resolved.parquet`:** Complete universe of 398 resolved establishments.
4. **Complete 8-table pack in `data/processed/tables/`**.
5. **Interactive Folium Resolution Map** with visual channel color coding and operational center.

In [ ]:
# Cell 2.5: Build Canonical Account Master, Validate Schema, Render Table 5 & Interactive Folium Map
import os
from pathlib import Path
import folium
from folium.plugins import MiniMap
import polars as pl
from IPython.display import HTML, display

# 1. Build Canonical Account Master (1 row per corporate parent entity)
canon_master = (
    act_stage2_sites.group_by("canonical_entity_id")
    .agg([
        pl.len().alias("site_count"),
        pl.col("canonical_parent_name").first().alias("canonical_parent_name"),
        pl.col("business_name").first().alias("primary_site_name"),
        pl.col("b2b_status").first().alias("b2b_status"),
        pl.col("is_rescued_partial").first().alias("is_rescued_partial"),
        pl.col("b2b_5d_score").max().alias("max_b2b_5d_score"),
        pl.col("total_est_employees_midpoint").first().alias("total_est_employees_midpoint"),
        pl.col("campus_consolidated_payroll_mxn").first().alias("consolidated_payroll_mxn"),
        pl.col("actionable_channel").first().alias("best_contact_channel"),
        pl.col("effective_contact").first().alias("effective_contact"),
        pl.col("email_source").first().alias("email_source"),
        pl.col("email_confidence").first().alias("email_confidence"),
        pl.col("contact_verified").first().alias("contact_verified"),
        pl.col("contact_risk").first().alias("contact_risk"),
        pl.col("website").first().alias("website"),
        pl.col("phone").first().alias("phone"),
        pl.col("linkedin_search_url").first().alias("linkedin_url"),
        pl.col("workforce_academic_level").first().alias("workforce_academic_level"),
        pl.col("osm_verified_storefront").any().alias("has_osm_verified_site"),
    ])
    .sort("max_b2b_5d_score", descending=True)
)

# 2. Build Site Action List (Physical site routing)
site_action_list = act_stage2_sites.with_columns([
    pl.when(pl.col("is_duplicate") == False)
    .then(pl.lit("primary_site"))
    .otherwise(pl.lit("branch_secondary_site"))
    .alias("role_in_campus")
])

# 3. Production Contract QA Assertions
assert len(df_enriched) == 398, f"Universe mismatch: {len(df_enriched)} != 398"
assert len(canon_master) > 0 and len(canon_master) < len(site_action_list), f"Canonical accounts ({len(canon_master)}) must be strictly deduplicated vs sites ({len(site_action_list)})"
assert canon_master.filter(pl.col("b2b_status") == "no_target").height == 0, "No-target found in canonical action master"
assert canon_master.filter(pl.col("canonical_entity_id").is_null()).height == 0, "Null canonical ID found"

# 4. Export Artifacts to data/processed and tables/
for base_dir in [Path("data/processed"), Path("../data/processed"), Path("notebooks/data/processed")]:
    if base_dir.exists() or base_dir.parent.exists():
        base_dir.mkdir(parents=True, exist_ok=True)
        t_dir = base_dir / "tables"
        t_dir.mkdir(parents=True, exist_ok=True)
        
        df_enriched.write_parquet(base_dir / "entities_resolved.parquet")
        site_action_list.write_csv(base_dir / "b2b_stage2_actionable_master.csv")
        site_action_list.write_csv(t_dir / "stage2_site_action_list.csv")
        site_action_list.write_parquet(base_dir / "stage2_site_action_list.parquet")
        canon_master.write_csv(t_dir / "stage2_canonical_action_master.csv")
        canon_master.write_parquet(base_dir / "stage2_canonical_action_master.parquet")

print(f"💾 Exported Master Resolved Universe: entities_resolved.parquet ({len(df_enriched):,} records)")
print(f"💾 Exported Canonical Account Master: stage2_canonical_action_master.csv ({len(canon_master):,} unique corporate entities)")
print(f"💾 Exported Site Action List: stage2_site_action_list.csv ({len(site_action_list):,} operational sites)")
print(f"💾 Persisted 8-file audit/table pack in data/processed/tables/")

# 5. Render Table 5: Canonical Master Actionable Ranking (Top 25 Corporate Parents)
top_canon = canon_master.head(25)
t5_rows = []
for rank, r in enumerate(top_canon.iter_rows(named=True), start=1):
    payroll_fmt = f"${r['consolidated_payroll_mxn']/1e6:.2f}M" if r.get('consolidated_payroll_mxn') else "-"
    risk_color = "#15803d" if r['contact_risk'] == 'low' else ("#b45309" if r['contact_risk'] == 'medium' else "#b91c1c")
    conf_badge = f"<span style='background:#f1f5f9; color:{risk_color}; padding:2px 6px; border-radius:3px; font-weight:bold; font-size:10px;'>{r['email_confidence'].upper()}</span>"
    sites_tag = f"<span style='background:#e0e7ff; color:#3730a3; padding:1px 5px; border-radius:10px; font-size:10px; font-weight:bold;'>{r['site_count']}</span>" if r['site_count'] > 1 else "1"
    t5_rows.append(f"""
    <tr style="border-bottom: 1px solid #e2e8f0;">
        <td style="padding: 6px 8px; font-weight: bold; text-align: center; color: #64748b;">{rank}</td>
        <td style="padding: 6px 10px; font-weight: 600; color:#1e3a8a;">{r.get('canonical_parent_name')}</td>
        <td style="padding: 6px 8px; text-align: center;">{sites_tag}</td>
        <td style="padding: 6px 8px; text-align: center; font-weight: bold; color: #0284c7;">{float(r.get('max_b2b_5d_score') or 0.0):.2f}</td>
        <td style="padding: 6px 8px; font-size: 11px;"><span style="background:#e0f2fe; color:#0369a1; padding:2px 6px; border-radius:3px;">{r.get('best_contact_channel')}</span></td>
        <td style="padding: 6px 10px; font-family: monospace; font-size: 11px; color:#15803d;">{r.get('effective_contact')}</td>
        <td style="padding: 6px 8px; text-align: center;">{conf_badge}</td>
        <td style="padding: 6px 8px; text-align: right; font-weight: 600; color:#0f766e;">{payroll_fmt}</td>
        <td style="padding: 6px 8px; text-align: center;"><a href="{r.get('linkedin_url')}" target="_blank" style="color:#0a66c2; text-decoration:none; font-weight:bold;">🔍 LinkedIn &rarr;</a></td>
    </tr>
    """)

html_table_5 = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin: 15px 0;">
    <h4 style="margin: 0 0 8px 0; color: #0f172a;">🏆 Tabla 5: Maestro Canónico de Cuentas B2B Accionables (Top 25 Cuentas Deduplicadas)</h4>
    <table style="width: 100%; border-collapse: collapse; text-align: left; font-size: 11px; background: white; border: 1px solid #cbd5e1; border-radius: 6px; overflow: hidden;">
        <thead style="background: #0284c7; color: white;">
            <tr>
                <th style="padding: 8px 8px; text-align: center;">#</th>
                <th style="padding: 8px 10px;">Entidad Matriz Canónica</th>
                <th style="padding: 8px 8px; text-align: center;">Sitios</th>
                <th style="padding: 8px 8px; text-align: center;">Score Max</th>
                <th style="padding: 8px 8px;">Mejor Canal</th>
                <th style="padding: 8px 10px;">Contacto Efectivo</th>
                <th style="padding: 8px 8px; text-align: center;">Confianza</th>
                <th style="padding: 8px 8px; text-align: right;">Nómina Total</th>
                <th style="padding: 8px 8px; text-align: center;">Prospección ABM</th>
            </tr>
        </thead>
        <tbody>
            {''.join(t5_rows)}
        </tbody>
    </table>
    <p style="font-size: 11px; color: #64748b; margin-top: 5px;"><i>Nota: Cada fila representa exactamente un grupo empresarial único. Guardado en stage2_canonical_action_master.csv.</i></p>
</div>
"""
display(HTML(html_table_5))

# 6. Build Interactive Folium Resolution Map (Free OSM Tiles - Zero API Key Warning)
center_lat, center_lon = STAGE2_CONFIG["center_lat"], STAGE2_CONFIG["center_lon"]
m = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles=None)

folium.TileLayer(
    tiles="https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",
    attr="&copy; <a href='https://www.openstreetmap.org/copyright'>OpenStreetMap</a> contributors",
    name="🗺️ OpenStreetMap (Libre sin API Key)",
    overlay=False,
    control=True,
).add_to(m)

folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
    attr="&copy; OpenStreetMap contributors &copy; CARTO",
    name="🏙️ CARTO Positron Directo",
    overlay=False,
    control=True,
).add_to(m)

folium.LayerControl(position="topright").add_to(m)

folium.Marker(
    location=[center_lat, center_lon],
    popup="<b>📍 The Coffee — Bajío Hub</b><br>Centro Operativo OSINT",
    tooltip="📍 Centro OSINT: The Coffee Bajío Hub",
    icon=folium.Icon(color="red", icon="coffee", prefix="fa"),
).add_to(m)

channel_colors = {
    "Direct Email (Verified DENUE)": "green",
    "Direct Email (Scraped Corporate Website)": "teal",
    "Campus Inherited Email (Parent Entity)": "blue",
    "Website / Synthesized Domain Email": "orange",
    "Direct Email (Rescued Tier-1 Corporate)": "purple",
    "Direct Phone + LinkedIn ABM": "darkblue",
    "Field Sales + LinkedIn ABM": "gray",
}

for r in site_action_list.iter_rows(named=True):
    ch = r.get("actionable_channel", "")
    color = channel_colors.get(ch, "cadetblue")
    popup_html = f"""
    <div style='font-family: Arial; font-size: 11px; min-width: 200px;'>
        <b style='color:#0f172a; font-size:12px;'>{r.get('business_name')}</b><br>
        <span style='color:#64748b;'>Matriz: {r.get('canonical_parent_name')}</span><hr style='margin:4px 0;'>
        <b>Score 5D:</b> {float(r.get('b2b_5d_score') or 0.0):.2f} pts<br>
        <b>Canal:</b> {ch}<br>
        <b>Contacto:</b> {r.get('effective_contact')} ({r.get('email_confidence', '').upper()})<br>
        <a href='{r.get('linkedin_search_url')}' target='_blank' style='color:#0a66c2; font-weight:bold;'>LinkedIn Search &rarr;</a>
    </div>
    """
    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=6 if r.get("campus_site_count", 1) > 1 else 4,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{r.get('business_name')} ({ch})",
    ).add_to(m)

for d in [Path('data/processed'), Path('../data/processed'), Path('notebooks/data/processed')]:
    if d.exists(): m.save(str(d / 'entities_resolved_map.html'))

MiniMap(toggle_display=True).add_to(m)
display(m)
